# Entity Reuse Processing


In [ ]:
import pandas as pd
import numpy as np
import pyarrow
from pathlib import Path
from collections import defaultdict, deque
from tqdm import tqdm
from datetime import datetime

In [ ]:
# Parameters
folder_output_path = Path("../results/Labeled_Datasets/Ecuador/Ecuador_part_all/2026_01_10")
df_path = folder_output_path / "processed_Ecuador_part_all.gzip.parquet"

In [ ]:
output_df_path = folder_output_path / "processed_Ecuador_part_all_TEST.gzip.parquet"

print(f"Output folder: {output_df_path}")

In [ ]:
# Load data
df = pd.read_parquet(df_path)
print(f"Loaded df from: {df_path}")
print(f"Loaded df with shape: {df.shape}")

In [ ]:
# write_report

REPORT_PATH = folder_output_path / "progress_report.txt"

def write_report(msg: str):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"{timestamp} | {msg}"
    print(line)
    with open(REPORT_PATH, "a") as f:
        f.write(line + "\n")

write_report(f"Started entity reuse processing for {df_path}")

In [ ]:
def add_entity_reuse(
    df: pd.DataFrame,
    time_window: str = "4D",
    entity_column: str = "hashtags",
    post_id_column: str = "postid",
    account_id_column: str = "accountid",
    timestamp_column: str = "post_time",
) -> None:
    """
    Adds temporal entity-reuse columns to a DataFrame.

    For each post, this function finds earlier posts within a `time_window` that share at least one entity.
    It adds columns with the post IDs and account IDs of these related posts.

    It maintains per-entity sliding windows to efficiently track recent usage,
    collects matching prior posts, and stores their post IDs and account IDs in new columns.
    It use dict{entity: deque[(timestamp, post_index)]} to track recent posts for each entity, expiring old entries as it goes.

    Note 1: An authors can reuse entities from their own earlier posts, and these will be included in the results. You can later set to ignore self-reuse when building the graph.
    Note 2: The function does not count for a post that reuses the same entity type multiple times between posts. It will only be counted once per post.

    Args:
        df (pd.DataFrame): The input DataFrame.
        entity_column (str): The column with entities for each post.
        time_window (str): The sliding window duration (default: "4D").
        post_id_column (str): The column with post identifiers (default: "postid").
        account_id_column (str): The column with account identifiers (default: "accountid").
        timestamp_column (str): The column with post timestamps (default: "post_time").

    Returns:
        None: Modifies the DataFrame in-place, adding:
              - f"{entity_column}_reuse_{time_window}_postids"
              - f"{entity_column}_reuse_{time_window}_accountids"
    """

    write_report(f"\n[add_entity_reuse] Adding entity reuse columns for '{entity_column}' with time window {time_window}...")

    # 1. Define output column names
    output_post_ids_column = f"{entity_column}_reuse_{time_window}_postids"
    output_account_ids_column  = f"{entity_column}_reuse_{time_window}_accountids"
    
    # Drop existing columns if they already exist to avoid conflicts
    existing_cols_to_drop = [col for col in [output_post_ids_column, output_account_ids_column] if col in df.columns]
    if existing_cols_to_drop:
        write_report(f"Dropping existing columns: {existing_cols_to_drop}")
        df.drop(columns=existing_cols_to_drop, inplace=True)

    # 2. Convert columns to NumPy arrays for performance
    timestamps = pd.to_datetime(df[timestamp_column]).to_numpy(dtype="datetime64[ns]")
    post_ids = df[post_id_column].to_numpy()
    account_ids = df[account_id_column].to_numpy()
    entities_per_post = df[entity_column].to_numpy()

    # Convert time window to numpy timedelta
    time_window_delta = np.timedelta64(pd.Timedelta(time_window).value, "ns")

    # Pre-allocate output lists
    num_posts = len(df)
    output_post_ids = [[] for _ in range(num_posts)]
    output_account_ids  = [[] for _ in range(num_posts)]

    # 3. Set up entity history tracker: {entity: deque[(timestamp, post_index)]}
    entities_history: dict[str, deque] = defaultdict(deque)

    # 4. Process posts in chronological order
    sorted_timestamp_indices = np.argsort(timestamps)
    for current_post_index in tqdm(sorted_timestamp_indices, desc="Processing posts"):
        current_timestamp = timestamps[current_post_index]
        current_entities = entities_per_post[current_post_index]

        # no entities in this post
        if current_entities is None or (len(current_entities) == 0):
            continue

        matching_post_indices = set()

        # 5. Find related earlier posts for each entity
        for entity in current_entities:
            
            # no entity
            if not entity:
                continue

            entity_post_deque = entities_history[entity] # deque: [(timestamp, post_index)] of post with this entity
            
            # Expire old posts from the deque
            while entity_post_deque and (current_timestamp - entity_post_deque[0][0]) > time_window_delta:
                entity_post_deque.popleft() # remove posts outside the time window

            # Collect indices of posts still in the window
            matching_post_indices.update(post_index for _, post_index in entity_post_deque)

        # 6. Store results in df
        if matching_post_indices:
            sorted_matching_indices = sorted(list(matching_post_indices), key=lambda index: timestamps[index]) # label order by timestamps, oldest is first

            # Remove duplicates while preserving order
            output_post_ids[current_post_index] = list(dict.fromkeys([post_ids[post_index] for post_index in sorted_matching_indices]))
            output_account_ids[current_post_index] = list(dict.fromkeys([account_ids[post_index] for post_index in sorted_matching_indices]))

        # 7. Add current post to history (after matching)
        for entity in current_entities:
            if entity:
                entities_history[entity].append((current_timestamp, current_post_index))

    # 8. Convert outputs to ndarrays and add to df
    output_post_ids = [np.asarray(x, dtype=object) for x in output_post_ids]
    output_account_ids = [np.asarray(x, dtype=object) for x in output_account_ids]
    
    df[output_post_ids_column] = output_post_ids
    print(f"[add_entity_reuse] Added column: {output_post_ids_column}")
    df[output_account_ids_column]  = output_account_ids
    print(f"[add_entity_reuse] Added column: {output_account_ids_column}")

In [ ]:
# Configuration for entity reuse
# ENTITY_COLS = ["hashtags", "urls", "account_mentions", "ner_entities"]
# time_windows = ["1W", "4D", "12H"] # See pandas.TimeDelta documentation. Example: "1W" for 1 week, "4D" for 4 days, "12H" for 12 hours, etc.


#  for test
ENTITY_COLS = ["hashtags"]
time_windows = ["4D"] 

In [ ]:
# Execute entity reuse processing
for time_window in time_windows:
    for entity_col in ENTITY_COLS:
        add_entity_reuse(df, time_window=time_window, entity_column=entity_col)

write_report(f"Entity reuse processing completed for time windows: {time_windows}")

In [ ]:
# Save results
df.to_parquet(output_df_path, index=False, compression="gzip")
write_report(f"Saved results to: {output_df_path}")
write_report(f"✓ Entity reuse processing complete!")